### The embedding table should follow the structure below in your BigQuery environment so the code works as is.

| Field Name  | Mode     | Type   | Description |
| ----------- | -------- | ------ | ----------- |
| doc_id      | NULLABLE | STRING |             |
| unique_id   | NULLABLE | STRING |             |
| content     | NULLABLE | STRING |             |
| embedding   | REPEATED | FLOAT  |             |
| metadata    | NULLABLE | STRING |             |
| company_id  | NULLABLE | STRING |             |
| active_flag | NULLABLE | STRING |             |


### example of create table query...

CREATE TABLE IF NOT EXISTS `your_project.your_dataset.embeddings_table`
(
  doc_id      STRING  OPTIONS(description = 'Document ID'),                                   
  unique_id   STRING  OPTIONS(description = 'External unique identifier'),                    
  content     STRING  OPTIONS(description = 'Raw text content'),                              
  embedding   ARRAY<FLOAT64> OPTIONS(description = 'Vector embedding as repeated floats'),    
  metadata    STRING  OPTIONS(description = 'Serialized metadata (JSON/string)'),             
  company_id  STRING  OPTIONS(description = 'Company identifier'),                            
  active_flag STRING  OPTIONS(description = 'Activation flag, e.g., Y/N')                     
)
OPTIONS(
  description = 'Embeddings storage with text, vector, and metadata'
);


In [ ]:
from image_summary import extract_text_from_image
import time
import asyncio
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import os
os.makedirs('image_test2', exist_ok=True)


class IngestionFlow:
    def __init__(self, company_id, doc_id, service_credentials, active_flag="Y"):
        self.company_id = company_id
        self.doc_id = doc_id
        self.service_credentials = service_credentials
        self.active_flag = active_flag

    def pdf_to_text_image_information_extraction(self, google_model_api, model, gs_location):
        import os
        import json
        import fitz  # PyMuPDF
        from google.cloud import storage
        from google.oauth2 import service_account
        from langchain_community.document_loaders import PyMuPDFLoader
        from langchain_community.document_loaders.parsers import LLMImageBlobParser
        from langchain_google_genai import ChatGoogleGenerativeAI
        from langchain.schema import Document

        os.environ["GOOGLE_API_KEY"] = google_model_api

        # Setup Google Cloud Storage client
        key_dict = self.service_credentials
        credentials = service_account.Credentials.from_service_account_info(key_dict)
        storage_client = storage.Client(credentials=credentials, project=key_dict["project_id"])

        # Parse GCS location
        gcs_uri = gs_location
        if gcs_uri.startswith("gs://"):
            gcs_uri = gcs_uri[len("gs://"):]
        bucket_name, blob_name = gcs_uri.split("/", 1)

        bucket = storage_client.bucket(bucket_name)
        blob = bucket.blob(blob_name)

        # Download PDF to local temp path
        original_file_name = os.path.basename(blob_name)
        temp_dir = "tmp"
        os.makedirs(temp_dir, exist_ok=True)
        temp_file_path = os.path.join(temp_dir, original_file_name)
        blob.download_to_filename(temp_file_path)
        print(f"Downloaded {blob_name} to {temp_file_path}")

        # Initialize Gemini Vision model
        llm = ChatGoogleGenerativeAI(
            model=model,
            max_output_tokens=8192,
            temperature=0
        )

        # PDF loader to extract text
        def load_pdf_func(file_path):
            return PyMuPDFLoader(
                file_path=file_path,
                images_inner_format=['text', 'markdown-img', 'html-img'],
                extract_tables='markdown',
                images_parser=LLMImageBlobParser(model=llm),
                extract_images=True
            )

        def load_pdf_with_retries(file_path, max_retries=3, base_delay=2):
            retries = 0
            while retries < max_retries:
                try:
                    loader = load_pdf_func(file_path)
                    docs = loader.load()
                    return docs
                except Exception as e:
                    error_msg = str(e)
                    if "503" in error_msg or "model is overloaded" in error_msg.lower():
                        wait_time = base_delay * (2 ** retries)
                        print(f"Model overloaded, retrying in {wait_time} seconds... (attempt {retries + 1}/{max_retries})")
                        time.sleep(wait_time)
                        retries += 1
                    else:
                        raise
            raise RuntimeError("Max retries exceeded due to model overload.")

        docs = load_pdf_with_retries(temp_file_path)
        page_to_image_convert = os.environ.get("page_to_image_convert", "false").lower() == "true"

        if page_to_image_convert and Path(temp_file_path).suffix.lower() == '.pdf':
            valid_docs = []
            pdf_page_char_len = int(os.environ.get("pdf_page_char_len", 0))

            with fitz.open(temp_file_path) as pdf_doc:
                pages_needing_ocr = []
                pages_with_text = []

                for i, doc in enumerate(docs):
                    page_text = doc.page_content if doc.page_content else ""
                    if (len(page_text.strip()) <= pdf_page_char_len or "no text" in page_text.lower() or not page_text.strip()):
                        pages_needing_ocr.append((i, doc))
                    else:
                        if not page_text.strip():
                            page_text = f"[Empty page {i + 1}]"
                        pages_with_text.append((i, doc, page_text))

                def process_page_image(page_info):
                    i, doc = page_info
                    try:
                        zoom = 2
                        mat = fitz.Matrix(zoom, zoom)
                        page = pdf_doc.load_page(i)
                        pix = page.get_pixmap(matrix=mat, dpi=300)
                        temp_image_path = os.path.join(temp_dir, f"page_{i + 1}.png")
                        test_image_path = os.path.join('image_test2', f"test_page_{i + 1}.png")
                        pix.save(temp_image_path)
                        pix.save(test_image_path)

                        try:
                            summary_text = extract_text_from_image(temp_image_path, google_model_api, model)
                            if not summary_text or not isinstance(summary_text, str) or not summary_text.strip():
                                summary_text = f"[No text extracted from page {i + 1}]"
                                print(f"Warning: Empty text extracted from page {i + 1}")
                        except Exception as e:
                            print(f"Error extracting text from image on page {i + 1}: {e}")
                            summary_text = f"[Error extracting text from page {i + 1}]"
                        finally:
                            if os.path.exists(temp_image_path):
                                os.remove(temp_image_path)

                        metadata_json = json.dumps({
                            **doc.metadata,
                            "company_id": self.company_id,
                            "doc_id": self.doc_id,
                            "active_flag": self.active_flag,
                            "page_to_image_convert": "Y"
                        })

                        return (i, Document(page_content=summary_text, metadata={"metadata": metadata_json}))
                    except Exception as e:
                        print(f"Failed to process page {i + 1}: {e}")
                        return None

                max_workers = min(2, len(pages_needing_ocr)) if pages_needing_ocr else 1
                ocr_results = []
                if pages_needing_ocr:
                    print(f"Processing {len(pages_needing_ocr)} pages with OCR using {max_workers} workers")
                    with ThreadPoolExecutor(max_workers=max_workers) as executor:
                        futures = [executor.submit(process_page_image, page_info) for page_info in pages_needing_ocr]
                        for future in as_completed(futures):
                            result = future.result()
                            if result:
                                ocr_results.append(result)

                text_results = []
                for i, doc, page_text in pages_with_text:
                    metadata_json = json.dumps({
                        **doc.metadata,
                        "company_id": self.company_id,
                        "doc_id": self.doc_id,
                        "active_flag": self.active_flag
                    })
                    text_results.append((i, Document(page_content=page_text, metadata={"metadata": metadata_json})))

                all_results = ocr_results + text_results
                all_results.sort(key=lambda x: x[0])
                valid_docs = [doc for _, doc in all_results]

            if os.path.exists(temp_file_path):
                try:
                    os.remove(temp_file_path)
                except PermissionError as e:
                    print(f"Warning: Could not delete temp PDF file: {e}")

            print(f"Created {len(valid_docs)} valid documents")
            return valid_docs
        else:
            for doc in docs:
                doc.metadata['source'] = gs_location
                doc.metadata['file_path'] = gs_location

            valid_docs = []
            for doc in docs:
                if doc.page_content and isinstance(doc.page_content, str) and doc.page_content.strip():
                    metadata_json = json.dumps({
                        **doc.metadata,
                        "company_id": self.company_id,
                        "doc_id": self.doc_id,
                        "active_flag": self.active_flag
                    })
                    valid_docs.append(
                        Document(
                            page_content=doc.page_content,
                            metadata={"metadata": metadata_json}
                        )
                    )
                else:
                    print("Warning: Skipping document with empty page_content")

            if os.path.exists(temp_file_path):
                os.remove(temp_file_path)

            print(f"Created {len(valid_docs)} valid documents")
            return valid_docs

    def embed_using_service_account_dict(self, embedding_model):
        from langchain_google_vertexai import VertexAIEmbeddings
        import google.oauth2.service_account

        key_dict = self.service_credentials
        PROJECT_ID = key_dict["project_id"]
        credentials = google.oauth2.service_account.Credentials.from_service_account_info(key_dict)
        print("Credentials object created successfully from dictionary.")

        embedding = VertexAIEmbeddings(
            model_name=embedding_model,
            project=PROJECT_ID,
            credentials=credentials
        )
        return embedding

    def bqvectore_store(self, dataset_id, table_id, REGION, embedding):
        from langchain_google_community import BigQueryVectorStore
        import google.oauth2.service_account

        key_dict = self.service_credentials
        PROJECT_ID = key_dict["project_id"]
        credentials = google.oauth2.service_account.Credentials.from_service_account_info(key_dict)
        print("Credentials object created successfully from dictionary.")

        bq_client = BigQueryVectorStore(
            project_id=PROJECT_ID,
            dataset_name=dataset_id,
            table_name=table_id,
            location=REGION,
            embedding=embedding,
            doc_id_field="unique_id",
            distance_type="COSINE",
            credentials=credentials
        )
        return bq_client

    def updated_doc_data(self, dataset_id, table_id):
        try:
            time.sleep(1)
            import traceback
            from bq_utill import BigQueryUtils
            key_dict = self.service_credentials
            bq_utils = BigQueryUtils(json_key_dict=key_dict)

            update_query = f"""
                UPDATE `{key_dict["project_id"]}.{dataset_id}.{table_id}`
                SET company_id='{self.company_id}', doc_id='{self.doc_id}', active_flag='{self.active_flag}'
                WHERE JSON_EXTRACT_SCALAR(metadata, '$.doc_id') = '{self.doc_id}';
            """
            print(f"Executing update query: {update_query}")
            bq_utils.execute_qry(update_query, 'dml')
            return "Successfully updated data."
        except Exception as e:
            print(f"Error in updated data: {e}")
            print(traceback.format_exc())
            return f"error: {str(e)}"

    def deactivate_old_embeddings(self, dataset_id, table_id):
        try:
            time.sleep(1)
            from bq_utill import BigQueryUtils
            key_dict = self.service_credentials
            bq_utils = BigQueryUtils(json_key_dict=key_dict)

            deactivate_query = f"""
                UPDATE `{key_dict["project_id"]}.{dataset_id}.{table_id}`
                SET active_flag = 'N'
                WHERE JSON_EXTRACT_SCALAR(metadata, '$.doc_id') = '{self.doc_id}' 
                AND active_flag = 'Y';
            """
            print(f"Deactivating old embeddings for doc_id: {self.doc_id}")
            bq_utils.execute_qry(deactivate_query, 'dml')
            print(f"Successfully deactivated old embeddings for doc_id: {self.doc_id}")
            return "Successfully deactivated old embeddings."
        except Exception as e:
            print(f"Error in deactivating old embeddings: {e}")
            return f"error: {str(e)}"

    # -------- Token estimation and batching helpers --------
    @staticmethod
    def _estimate_tokens_for_text(text: str) -> int:
        """
        Rough token estimate: ~4 chars per token for Gemini.
        Cap per-input at 2048 tokens as per Vertex embeddings limits.
        """
        if not text:
            return 0
        approx = max(1, int(len(text) / 4))
        return min(approx, 2048)

    def batch_pages(self, docs, max_pages_per_batch: int = 20, max_tokens_per_batch: int = 19000):
        """
        Build batches by pages:
        - No batch exceeds max_pages_per_batch pages
        - Token-aware: do not exceed max_tokens_per_batch total estimated tokens
        """
        batches = []
        current = []
        token_sum = 0

        for d in docs:
            page_tokens = self._estimate_tokens_for_text(getattr(d, "page_content", "") or "")
            
            # If adding this page exceeds either limit, flush current batch
            if current and (len(current) >= max_pages_per_batch or token_sum + page_tokens > max_tokens_per_batch):
                batches.append(current)
                current = []
                token_sum = 0

            current.append(d)
            token_sum += page_tokens

        if current:
            batches.append(current)
        
        return batches

    @staticmethod
    def _is_token_limit_error(error_msg: str) -> bool:
        """Detect if error is related to token limits"""
        error_lower = error_msg.lower()
        return any(keyword in error_lower for keyword in [
            'token', 'limit', '400', 'exceed', 'supports up to', 'input token count'
        ])

    def _insert_batch_with_adaptive_retry(
        self, 
        bq_client, 
        batch, 
        batch_idx: int, 
        initial_pages_per_batch: int,
        max_retries: int = 10
    ):
        """
        Insert a batch with automatic page reduction on token errors.
        Tries up to max_retries times, reducing pages by 1 each attempt.
        
        Returns: (success_count, failed_pages)
        """
        pages_per_batch = initial_pages_per_batch
        retry_count = 0
        remaining_docs = batch.copy()
        total_inserted = 0
        failed_pages = []

        while remaining_docs and retry_count < max_retries:
            try:
                # Calculate current batch size based on reduced pages_per_batch
                current_batch_size = min(pages_per_batch, len(remaining_docs))
                current_batch = remaining_docs[:current_batch_size]
                
                print(f"  → Attempt {retry_count + 1}: Trying {len(current_batch)} pages (limit: {pages_per_batch} pages/batch)")
                
                # Try to insert
                bq_client.add_documents(current_batch)
                
                # Success!
                total_inserted += len(current_batch)
                print(f"  ✓ Successfully inserted {len(current_batch)} pages")
                
                # Remove successfully inserted docs from remaining
                remaining_docs = remaining_docs[current_batch_size:]
                
                # Reset retry count for next sub-batch
                retry_count = 0
                pages_per_batch = initial_pages_per_batch  # Reset to original limit
                
                # Small delay between sub-batches
                if remaining_docs:
                    time.sleep(0.3)
                
            except Exception as e:
                error_msg = str(e)
                
                # Check if it's a token limit error
                if self._is_token_limit_error(error_msg):
                    retry_count += 1
                    
                    if retry_count >= max_retries:
                        print(f"  ✗ Max retries ({max_retries}) reached for batch {batch_idx}")
                        print(f"  ✗ Failed to insert {len(remaining_docs)} pages after trying down to {pages_per_batch} pages/batch")
                        
                        # Try page-by-page as last resort
                        print(f"  → Last resort: Trying page-by-page insertion for {len(remaining_docs)} pages")
                        for single_doc in remaining_docs:
                            try:
                                bq_client.add_documents([single_doc])
                                total_inserted += 1
                                print(f"    ✓ Single page inserted")
                                time.sleep(0.2)
                            except Exception as single_error:
                                print(f"    ✗ Single page failed: {single_error}")
                                failed_pages.append(single_doc)
                        
                        break
                    
                    # Reduce pages per batch by 1
                    pages_per_batch = max(1, pages_per_batch - 1)
                    print(f"  ⚠ Token limit error detected. Reducing to {pages_per_batch} pages/batch (attempt {retry_count}/{max_retries})")
                    print(f"  ⚠ Error details: {error_msg[:200]}...")
                    
                    time.sleep(0.5)  # Brief pause before retry
                    
                else:
                    # Non-token error - try page-by-page
                    print(f"  ✗ Non-token error in batch {batch_idx}: {error_msg[:200]}")
                    print(f"  → Attempting page-by-page insertion for {len(remaining_docs)} pages")
                    
                    for single_doc in remaining_docs:
                        try:
                            bq_client.add_documents([single_doc])
                            total_inserted += 1
                            print(f"    ✓ Single page inserted")
                            time.sleep(0.2)
                        except Exception as single_error:
                            print(f"    ✗ Single page failed: {single_error}")
                            failed_pages.append(single_doc)
                    
                    break

        return total_inserted, failed_pages

    def final_ingestion(
        self,
        google_model_api,
        model,
        gs_location,
        embedding_model,
        dataset_id,
        table_id,
        REGION,
        is_reembed=False,
        max_pages_per_batch: int = 20,
        max_retry_attempts: int = 10
    ):
        """
        Complete ingestion with automatic batch size reduction on token errors.
        
        Args:
            max_pages_per_batch: Initial maximum pages per batch (default: 20)
            max_retry_attempts: Maximum retry attempts with reducing page count (default: 10)
        """
        try:
            start_time = time.time()

            if is_reembed:
                print(f"Re-embedding detected for doc_id: {self.doc_id}. Deactivating old embeddings...")
                self.deactivate_old_embeddings(dataset_id, table_id)

            # Extract documents
            extraction_start = time.time()
            docs = self.pdf_to_text_image_information_extraction(google_model_api, model, gs_location)
            extraction_time = time.time() - extraction_start
            print(f"Extraction completed in {extraction_time:.2f} seconds")

            if not docs:
                print("Warning: No valid documents to ingest")
                return "No valid documents to ingest"

            total_pages = len(docs)
            print(f"\n{'='*70}")
            print(f"Processing {total_pages} pages for ingestion")
            print(f"Initial batch size: {max_pages_per_batch} pages/batch")
            print(f"Auto-retry enabled: Up to {max_retry_attempts} attempts with adaptive page reduction")
            print(f"{'='*70}\n")

            # Build initial batches
            batches = self.batch_pages(docs, max_pages_per_batch=max_pages_per_batch, max_tokens_per_batch=19000)
            print(f"Created {len(batches)} initial batches\n")

            # Embedding client and vector store
            embed_start = time.time()
            embedding = self.embed_using_service_account_dict(embedding_model)
            bq_client = self.bqvectore_store(dataset_id, table_id, REGION, embedding)

            total_inserted = 0
            all_failed_pages = []

            # Process each batch with adaptive retry
            for idx, batch in enumerate(batches, start=1):
                print(f"\n--- Batch {idx}/{len(batches)} ({len(batch)} pages) ---")
                
                inserted_count, failed_pages = self._insert_batch_with_adaptive_retry(
                    bq_client=bq_client,
                    batch=batch,
                    batch_idx=idx,
                    initial_pages_per_batch=max_pages_per_batch,
                    max_retries=max_retry_attempts
                )
                
                total_inserted += inserted_count
                all_failed_pages.extend(failed_pages)
                
                print(f"Batch {idx} complete: {inserted_count}/{len(batch)} pages inserted")
                print(f"Running total: {total_inserted}/{total_pages} pages")
                
                # Brief pause between batches
                if idx < len(batches):
                    time.sleep(0.4)

            embed_time = time.time() - embed_start
            
            # Final summary
            print(f"\n{'='*70}")
            print(f"INGESTION SUMMARY")
            print(f"{'='*70}")
            print(f"Total pages processed: {total_pages}")
            print(f"Successfully inserted: {total_inserted}")
            print(f"Failed pages: {len(all_failed_pages)}")
            print(f"Success rate: {(total_inserted/total_pages)*100:.1f}%")
            print(f"Embedding and insertion time: {embed_time:.2f} seconds")
            print(f"{'='*70}\n")

            # Update metadata
            time.sleep(3)
            update_result = self.updated_doc_data(dataset_id, table_id)
            print(f"Metadata update result: {update_result}")

            total_time = time.time() - start_time
            print(f"\nTotal ingestion time: {total_time:.2f} seconds")
            
            if len(all_failed_pages) > 0:
                return f"Ingestion completed with warnings. Inserted {total_inserted}/{total_pages} pages. {len(all_failed_pages)} pages failed."
            else:
                return f"Ingestion completed successfully. Inserted {total_inserted}/{total_pages} pages."

        except Exception as e:
            import traceback
            print(f"\n{'='*70}")
            print(f"CRITICAL ERROR DURING INGESTION")
            print(f"{'='*70}")
            print(f"Error: {e}")
            print(traceback.format_exc())
            print(f"{'='*70}\n")
            return f"An error occurred: {e}"


In [ ]:
company_id = " which company id you want to add tag to the pdf embedding provide ther"
doc_id = "provide doc_id as unique identity to this document"
service_credentials = "provide here json key of gcp that key to permision off vertex ai biquery gcp bucket that json key place here."
import os
google_model_api= os.environ["gemini_api"]
model= os.environ["model_lite"]
# provide pdf/docx gs location 
gs_location = "gs://aaaaaaaa/bbbbbbbb/cccccccc.pdf"
embedding_model=os.environ["embedding_model"]
dataset_id= "provide here biqyuery dataset biquery of that embeding table in that available."
table_id= os.environ["pvt_data_emb_table"]
REGION = "us-central1" #provide region like this as per your dataset available.

In [ ]:
obj = IngestionFlow(company_id, doc_id, service_credentials, active_flag="Y")
result = obj.final_ingestion(google_model_api, model, gs_location, embedding_model, dataset_id, table_id, REGION, is_reembed=True)
print(result)